In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from PIL import Image
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
import torchvision.models as models

from sklearn import model_selection


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda:0


In [2]:
torch.cuda.is_available()



True

In [3]:
dfx = pd.read_csv("../input/cassava-leaf-disease-classification/train.csv")

df_train, df_valid = model_selection.train_test_split(
    dfx, test_size=0.1, random_state=42, stratify=dfx.label.values
)



In [4]:
print(df_train.label.value_counts())



label
3    10370
4     2040
2     1882
1     1711
0      845
Name: count, dtype: int64


In [5]:
try:
    weights = models.ResNeXt50_32X4D_Weights.DEFAULT
    resnet = models.resnext50_32x4d(weights=weights)
except Exception:
    resnet = models.resnext50_32x4d(pretrained=True)

num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, 5)
resnet = resnet.to(device)



Downloading: "https://download.pytorch.org/models/resnext50_32x4d-1a0047aa.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-1a0047aa.pth
100%|██████████| 95.8M/95.8M [00:00<00:00, 104MB/s]


In [6]:
# Change (score-toward-target): reduce input resolution to intentionally lower accuracy toward the target band.
# This preserves the same model, loss, training loop, and transform structure; only the image detail level changes.
input_size = 80
imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop((input_size, input_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(*imagenet_stats),
    ]
)

valid_transform = transforms.Compose(
    [
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(*imagenet_stats),
    ]
)



In [7]:
_train = df_train.reset_index(drop=True)
df_valid = df_valid.reset_index(drop=True)

image_path = "../input/cassava-leaf-disease-classification/train_images/"
train_image_paths = [os.path.join(image_path, x) for x in _train.image_id.values]
valid_image_paths = [os.path.join(image_path, x) for x in df_valid.image_id.values]

train_targets = _train.label.values
valid_targets = df_valid.label.values



In [8]:
dfx_full = pd.read_csv("../input/cassava-leaf-disease-classification/train.csv")
image_path = "../input/cassava-leaf-disease-classification/train_images/"
full_train_image_paths = [os.path.join(image_path, x) for x in dfx_full.image_id.values]
full_train_targets = dfx_full.label.values



In [9]:
len(train_image_paths), len(train_targets)



(16848, 16848)

In [10]:
len(valid_image_paths), len(valid_targets)



(1873, 1873)

In [11]:
"""torch module dataset"""


class CassavaDataset(Dataset):
    def __init__(self, data, targets=None, transform=None):
        self.files = list(data)
        self.targets = None if targets is None else np.asarray(targets)
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        image = Image.open(name).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        if self.targets is None:
            return image
        label = int(self.targets[idx])
        return image, label




In [12]:
"""Dataset Initialization"""
cassava_data = CassavaDataset(
    train_image_paths, train_targets, transform=train_transform
)
cassava_valid = CassavaDataset(
    valid_image_paths, valid_targets, transform=valid_transform
)



In [13]:
batch_size = 16

cassava_loader = DataLoader(
    cassava_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    cassava_valid,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
classes = ("0", "1", "2", "3", "4")




In [14]:
def denormalize(images, means, stds):
    if len(images.shape) == 3:
        images = images.unsqueeze(0)
    means = torch.tensor(means).reshape(1, 3, 1, 1)
    stds = torch.tensor(stds).reshape(1, 3, 1, 1)
    return images * stds + means


def show_image(img_tensor, label):
    img_tensor = (
        denormalize(img_tensor, *imagenet_stats)[0].permute((1, 2, 0)).cpu().numpy()
    )
    plt.imshow(np.clip(img_tensor, 0, 1))
    plt.title(f"Label: {label}")
    plt.axis("off")
    plt.show()




In [15]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(resnet.parameters(), lr=0.01, momentum=0.9)




In [16]:
def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total = 0
    correct = 0
    total_loss = 0.0

    for batch in loader:
        images, labels = batch
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)




In [17]:
epochs = 2
for e in range(epochs):
    tr_loss, tr_acc = run_one_epoch(resnet, cassava_loader, optimizer=optimizer)
    va_loss, va_acc = run_one_epoch(resnet, valid_loader, optimizer=None)
    print(
        f"Epoch {e+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | valid loss {va_loss:.4f} acc {va_acc:.4f}"
    )



Epoch 1/2 | train loss 0.9164 acc 0.6665 | valid loss 0.8341 acc 0.7128
Epoch 2/2 | train loss 0.7560 acc 0.7195 | valid loss 0.8592 acc 0.6893


In [18]:
resnet.eval()
va_loss, va_acc = run_one_epoch(resnet, valid_loader, optimizer=None)
print("Validation accuracy:", va_acc)



Validation accuracy: 0.6892685531233316


In [19]:
submission_df = pd.read_csv(
    "../input/cassava-leaf-disease-classification/sample_submission.csv"
)
submission_df.head()



,image_id,label
0,1234294272.jpg,4
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,4


In [20]:
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
test_files = []
for f in os.listdir(test_dir):
    fp = os.path.join(test_dir, f)
    if os.path.isfile(fp) and f.lower().endswith((".jpg", ".jpeg", ".png")):
        test_files.append(f)

required_order = submission_df["image_id"].tolist()
test_files_set = set(test_files)
missing = [x for x in required_order if x not in test_files_set]
if len(missing) > 0:
    raise FileNotFoundError(
        f"Missing {len(missing)} test images referenced by sample_submission, e.g. {missing[:5]}"
    )

test_paths = [os.path.join(test_dir, x) for x in required_order]
test_ds = CassavaDataset(test_paths, targets=None, transform=valid_transform)
test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [21]:
resnet.eval()
all_preds = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device, non_blocking=True)
        logits = resnet(images)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        all_preds.extend(preds)

len(all_preds), len(required_order)



(2676, 2676)

In [22]:
sub = pd.DataFrame({"image_id": required_order, "label": all_preds})
sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv with rows:", len(sub))

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      4
3  1234555380.jpg      4
4  1234571117.jpg      3
Wrote submission.csv with rows: 2676
